In [ ]:
# %%
# SECTION 1: Imports + Sample Data
# ----------------------------------
# Loads pandas (the only dependency) and writes a small sample CSV to disk
# so this notebook is self-contained and runnable without any external file.
# To use your own CSV, update input_path in Section 2 and skip the sample.
#
# Requirements: pip install pandas

import pandas as pd

# Write a self-contained sample CSV for demonstration
sample = pd.DataFrame({
    'id':      [1, 2, 3, 4, 4, 5],
    'name':    ['Alice', 'Bob', 'Carol', 'Dave', 'Dave', 'Eve'],
    'country': ['US', 'CA', 'US', 'CA', 'CA', 'US'],
    'status':  ['active', 'inactive', 'active', 'active', 'active', 'inactive'],
    'price':   [120.0, 80.0, 200.0, 150.0, 150.0, 90.0],
})
sample.to_csv('sample.csv', index=False)

print("Imports loaded. sample.csv written.")
sample

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your transformation options here before running any section below.
# Operations always apply in this fixed order: filter → dedupe → rename → select.
#
# input_path   - Path to the source CSV file
# filter_expr  - pandas query() expression (None to skip), e.g. "price > 100"
# dedupe       - True to drop duplicate rows
# dedupe_cols  - Deduplicate on a column subset (list or None for all columns)
# rename_map   - Dict of {old_name: new_name} column renames (empty dict to skip)
# select_cols  - List of columns to keep in order (None to keep all)
# output_path  - Where to save the resulting CSV

input_path  = 'sample.csv'
filter_expr = "status == 'active'"
dedupe      = True
dedupe_cols = ['id']           # deduplicate on id only; None = all columns
rename_map  = {'name': 'full_name', 'price': 'unit_price'}
select_cols = ['id', 'full_name', 'country', 'unit_price']
output_path = 'output.csv'

print(f"Config ready. Input: {input_path}")

In [ ]:
# %%
# SECTION 3: load_csv()
# -----------------------
# Reads the CSV at input_path into a pandas DataFrame.
# Prints the shape and data types so you can confirm column names and types
# before applying transformations.

import pathlib

def load_csv(path: str) -> pd.DataFrame:
    if not pathlib.Path(path).exists():
        raise FileNotFoundError(f"File not found: '{path}'")
    df = pd.read_csv(path)
    print(f"Loaded '{path}': {df.shape[0]} rows x {df.shape[1]} cols")
    return df

df = load_csv(input_path)
print(f"\nColumn types:\n{df.dtypes}")
df.head()

In [ ]:
# %%
# SECTION 4: apply_filter()
# --------------------------
# Filters rows using a pandas query() expression set in filter_expr.
# Prints the row count before and after so you can see how many rows
# are removed. If the expression is invalid, a warning is printed and
# the original DataFrame is returned unchanged.
# Skip this cell (or set filter_expr = None) if no filtering is needed.

def apply_filter(df: pd.DataFrame, expr: str) -> pd.DataFrame:
    before = len(df)
    try:
        df = df.query(expr)
    except Exception as e:
        print(f"Warning: filter expression failed ({e}). No filter applied.")
        return df
    print(f"Filter: {before} → {len(df)} rows")
    return df

if filter_expr:
    df = apply_filter(df, filter_expr)
df.head()

In [ ]:
# %%
# SECTION 5: apply_deduplication()
# ----------------------------------
# Drops duplicate rows from the DataFrame. When dedupe_cols is set, only
# considers those columns when identifying duplicates (useful for deduplicating
# on a key like 'id' while keeping other columns intact).
# Prints how many rows were removed.

def apply_deduplication(df: pd.DataFrame, subset) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=subset if subset else None)
    print(f"Dedupe: removed {before - len(df)} duplicate row(s), {len(df)} remaining")
    return df

if dedupe:
    df = apply_deduplication(df, dedupe_cols)
df.head()

In [ ]:
# %%
# SECTION 6: apply_rename()
# --------------------------
# Renames columns according to rename_map (set in Section 2).
# Warns if any key in rename_map does not match an existing column name.
# Shows column names before and after so you can confirm the rename worked.

def apply_rename(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    unknown = [k for k in mapping if k not in df.columns]
    if unknown:
        print(f"Warning: rename skipped for unknown column(s): {unknown}")
    return df.rename(columns=mapping)

if rename_map:
    print(f"Before: {list(df.columns)}")
    df = apply_rename(df, rename_map)
    print(f"After:  {list(df.columns)}")
df.head()

In [ ]:
# %%
# SECTION 7: apply_column_selection()
# -------------------------------------
# Keeps only the columns listed in select_cols, in the order given.
# Warns about any column names that don't exist (after renaming) and
# keeps only the valid intersection. Shows the final column list.

def apply_column_selection(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        print(f"Warning: column(s) not found and skipped: {missing}")
    keep = [c for c in columns if c in df.columns]
    return df[keep]

if select_cols:
    df = apply_column_selection(df, select_cols)
    print(f"Selected columns: {list(df.columns)}")
df.head()

In [ ]:
# %%
# SECTION 8: Inspect result
# --------------------------
# Shows the final shape and a full preview of the transformed DataFrame.
# Review this before writing to disk. If the result looks wrong, adjust
# the configuration in Section 2 and re-run the relevant sections.

print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"Columns: {list(df.columns)}")
df

In [ ]:
# %%
# SECTION 9: write_csv()
# -----------------------
# Writes the final transformed DataFrame to output_path (set in Section 2).
# Run this last, after confirming the Section 8 preview looks correct.

def write_csv(df: pd.DataFrame, output_path: str) -> None:
    df.to_csv(output_path, index=False)
    print(f"Wrote {len(df)} rows to '{output_path}'")

write_csv(df, output_path)